In [ ]:
Istruzioni



CHIUDERE IL FILE CHE SI VUOLE CALCOLARE PER UN PAIO DI MINUTI

NON CHIUDERE IL PROGRAMMA, SOLO MINIMIZZARE LO SCHERMO



MODIFICA CODICE E FILTRI

#Importare cartelle:
#il nome dei file da calcolare devono avere il nome in questo formato

#esempio:
_lista_transazioni_03_2026
_lista_transazioni_11_2028
_lista_transazioni_06_2032



#Filtri: 
1Inserire i dati necessari nello stesso formato che ci sono già in precendeza            
filtri_chiusura_weekend   filtro_no_pay

2fermare il programma 
3premere restart 
4clic nel triangolo (su a sinistra) per farlo ripartire


#recupero files
"Z:\path\cartella_sicurezza"



Troubleshooting 

#MESSAGGIO ERRORE:
IL FILE è ANCORA APERTO SU PYTHON

#1 CHIUDERE E RIAPRIRE PYTHON E FAR PARTIRE IL PROGRAMMA

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#MESSAGGIO ERRORE:
PermissionError: [Errno 13] Permission denied: 'Z:/path/_lista_transazioni_02_2026.xlsx'   o simile


#1 CONTROLLARE CHE I FILE EXCEL NON SIANO APERTI E FAR RIPARTIRE IL CODICE, aspetta 2 minuti e finalmente si possono riaprire i file
#2 CONTROLLARE I PERMESSI DEL PC CHE FA GIRARE IL PROGRAMMA

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#MESSAGGIO ERRORE:
BadZipFile: File is not a zip file

#1 eliminare file segnalato come zip corrotto e scambiare per quello dentro la cartella "cartella_sicurezza"
# e far ripartire il programma

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------
#MESSAGGIO ERRORE:
FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Z:/path/Telepass'

#1 aprire il programma CloudDisk dal desktop 
#e far vpartire il programma

In [ ]:
import pandas as pd
import os
import openpyxl
import time
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")
from pandasql import sqldf


#impostare i giorni chiusura e festivi per ogni mese in ogni anno
filtri_chiusura_weekend={
            '_lista_transazioni_01_2026.xlsx':["02/01/26","01/01/26","05/01/26"],
            '_lista_transazioni_02_2026.xlsx':["06/02/26"],
            '_lista_transazioni_03_2026.xlsx':[],
            '_lista_transazioni_04_2026.xlsx':["06/04/26"],
            '_lista_transazioni_05_2026.xlsx':[],
            '_lista_transazioni_06_2026.xlsx':[],
            '_lista_transazioni_07_2026.xlsx':[],
            '_lista_transazioni_08_2026.xlsx':[],
            '_lista_transazioni_09_2026.xlsx':[],
            '_lista_transazioni_10_2026.xlsx':[],
            '_lista_transazioni_11_2026.xlsx':[],
            '_lista_transazioni_12_2026.xlsx':[]
            }


#agenti da non addebitare
filtro_no_pay=[
            "persona_1",
            "perosna_2"
            ]





cartella_file="Z:/path/Telepass/"

#cartella con file di backup
cartella_sicurezza="Z:/path/cartella_sicurezza/"

def pulizia_df(tutto,mese):
    #'Data'

    tutto['Data']=pd.to_datetime(tutto['Data'],errors='coerce',dayfirst=True).dt.strftime(f"%d/%m/%y")
    tutto["Data"]=tutto["Data"].astype(str)

    lista_gm=[]

    #aggiustamento giorno e mese
    for gm in tutto["Data"]:
        if (gm.split("/")[0]) == mese:
            lista_gm.append((gm.split("/")[1])+"/"+(gm.split("/")[0])+"/"+(gm.split("/")[2]))
        else:
            lista_gm.append(gm)

    tutto["Data"]=lista_gm

    lista_gm.clear()


    #'Importo'                   
    tutto['Importo']=tutto['Importo'].astype(str)
    tutto.loc[pd.notna(tutto['Importo']),'Importo']=tutto.loc[pd.notna(tutto['Importo']),'Importo'].str.replace(',','.') 
    tutto['Importo']=tutto['Importo'].astype(float)
    return tutto

#ESECUZIONE PROGRAMMA
while True:
    dirs=[]
    i_files=[]
    j_files=[]

    tutti_file=[f for f in os.listdir(cartella_file) if "_lista_transazioni_" not in f]
    sicurezza_files=[f for f in os.listdir(cartella_sicurezza) if "_lista_transazioni_" in f]

    #elenco cartelle da usare
    for i in tutti_file:
        a=os.path.join(f"{cartella_file}{i}")
        dirs.append(a)

    #path dir e files
    for i in dirs[1:]:
        i_files=[f for f in os.listdir(i) if "_lista_transazioni_" in f]


        #path di ogni file
        for j in i_files:
            
            #controllo se i file sono stati calcolati
            if j not in sicurezza_files:
                ####anche se il file è aperto, continua a lavorare
                try:
                    pathc=os.path.join(f"{i}/{j}")
                    df=pd.read_excel(pathc)
                    mese=j.split("_")[3]
                    df=pulizia_df(df,mese)
                    

                    #filtri chiusura weekend
                    df_noaz=df[df["Utilizzo"].isin(["Non Aziendale"])]
                    df_do_sa_az=df[df["Giorno"].isin(["domenica","sabato"]) & (df["Utilizzo"].isin(["Aziendale"]))]
                    
                    for cerca_filtro in filtri_chiusura_weekend.keys():
                        if j == cerca_filtro:

                            for filt in filtri_chiusura_weekend[j]:
                                filtraggio=[]
                                filtraggio.append(df[df["Utilizzo"].isin(["Aziendale"]) & (df["Data"] == (filt))])    
                            
                            if filtraggio:
                                df_festivo=pd.concat(filtraggio,ignore_index=True)    
                            filtraggio.clear()
                        
                    df_w_f_c=pd.concat([df_noaz,df_do_sa_az,df_festivo])
                    df_festivo=df_festivo.iloc[0:0]

                    


                    #filtri NO PAY
                    
                    for f in filtro_no_pay:
                        df_w_f_c=df_w_f_c[df_w_f_c["Dipendente"] != f]
                

                    query="""
                    SELECT "Codice fiscale", Dipendente,Sum(Importo) as "Importo"
                    FROM df_w_f_c
                    GROUP BY Dipendente
                    """

                    df_addebiti=sqldf(query,globals())
                    

                    #riga totale finale
                    df_total=pd.DataFrame({
                        "Codice fiscale":["Totale"],
                        "Dipendente":[""],
                        "Importo":[df_addebiti["Importo"].sum()]
                    })

                    df_addebiti=pd.concat([df_addebiti,df_total])



                    with pd.ExcelWriter(pathc) as writer:
                        
                        nome=j.replace(".xlsx","")
                        df.to_excel(writer,sheet_name=nome,index=False)

                        nome="weekend_festivo"
                        df_w_f_c.to_excel(writer,sheet_name=nome,index=False) 
                        

                        nome="Da addebitare"
                        df_addebiti.to_excel(writer,sheet_name=nome,index=False)
                        
                    
                    #creare un nuovo file excel e se esiste già sostituiscilo
                    file_sicurezza=os.path.join(f"{cartella_sicurezza}/{j}")
                    with pd.ExcelWriter(file_sicurezza) as writer:
                        nome=j.replace(".xlsx","")
                        df.to_excel(writer,sheet_name=nome,index=False)

                        nome="weekend_festivo"
                        df_w_f_c.to_excel(writer,sheet_name=nome,index=False) 
                

                        nome="Da addebitare"
                        df_addebiti.to_excel(writer,sheet_name=nome,index=False)

                    print(f"FILE:  {j}")

                    hour=datetime.now().strftime("%d/%m/%Y %H:%M:%S")
                    print(f"Aggiornati il: {hour}")
                    print("+"*55)


                    df_w_f_c=df_w_f_c.iloc[0:0]
                    df_addebiti=df_addebiti.iloc[0:0]
                    df_festivo=df_festivo.iloc[0:0]
                    df_do_sa_az=df_do_sa_az.iloc[0:0]
                    df_noaz=df_noaz.iloc[0:0]
                    df_total=df_total.iloc[0:0]
                except PermissionError:
                    hour=datetime.now().strftime("%d/%m/%Y %H:%M")
                    print(f"Per favore chiudere il file {j} per un paio di minuti {hour}")
                    continue
    time.sleep(120)